# Project 02: Intelligent Retail Demand Forecasting & Inventory Optimization

Run `generate_retail_data.py` once before the notebook. The reusable, documented pipeline implementation is in `retail_forecast_pipeline.py`; these cells run its stages in order. All paths are relative to the project folder, and outputs are written to `data/processed/`.

The validation is chronological: the earliest 70% of feature dates trains candidate models, the next 15% selects parameters, and the final 15% is an untouched test interval. The backtest is a daily rolling one-step-ahead forecast; actual prior-day sales, same-day promotion and price are treated as available inputs. This is not a fixed-origin multi-day forecast.

In [1]:
from pathlib import Path
import pandas as pd
from retail_forecast_pipeline import (
    RAW_DIR, PROCESSED_DIR, load_and_prepare, run_eda,
    calculate_statistics, add_features, fit_and_evaluate,
    export_inventory_risk, export_data_dictionary,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw input directory: {RAW_DIR}")
print(f"Processed output directory: {PROCESSED_DIR}")

Raw input directory: D:\cogniza internship data science project\project 2\data\raw
Processed output directory: D:\cogniza internship data science project\project 2\data\processed


## 1. Data ingestion and preparation

Required-column and file checks fail explicitly. Mixed date formats are normalized, duplicate store-product-date rows are removed, dimension relationships are validated, and absent dates are completed with zero units. Missing prices carry forward within each store/SKU, with product base price for leading gaps.

In [2]:
panel, inventory = load_and_prepare()
print(f"Completed daily panel: {len(panel):,} rows")
print(f"Duplicate observations removed: {panel.attrs.get('duplicate_rows_removed', 'see loader diagnostics')}")
display(panel.head())
display(inventory.head())

Completed daily panel: 365,000 rows
Duplicate observations removed: 3605


,date,store_id,product_id,units_sold,unit_price,promo_flag,category,subcategory,base_price,unit_cost,lead_time_days,city,store_size,region,sales_amount
0,2024-11-22,S01,P001,0,91.90,0,Grocery,Beverages,94.00,51.86,11,Seattle,Small,West,0.00
1,2024-11-22,S01,P002,17,84.58,0,Grocery,Snacks,85.20,38.19,19,Seattle,Small,West,1437.86
2,2024-11-22,S01,P003,2,115.32,0,Grocery,Pantry,117.20,75.99,16,Seattle,Small,West,230.64
3,2024-11-22,S01,P004,14,17.79,1,Grocery,Dairy,19.73,10.95,17,Seattle,Small,West,249.06
4,2024-11-22,S01,P005,17,47.60,0,Grocery,Frozen,47.64,33.25,17,Seattle,Small,West,809.20


,store_id,product_id,current_stock_on_hand,reorder_point
0,S01,P001,43,39
1,S01,P002,149,244
2,S01,P003,11,66
3,S01,P004,106,154
4,S01,P005,77,267


## 2. Exploratory analysis and 3. demand statistics

The EDA stage writes daily, weekly, monthly, category, promotion, and store charts to `data/processed/figures/`. Statistics include mean, median, standard deviation, quartiles, IQR, an IQR-based extreme-spike count, and product-level coefficients of variation.

In [3]:
run_eda(panel)
statistics, product_variability = calculate_statistics(panel)
display(statistics)
display(product_variability.sort_values('coefficient_of_variation', ascending=False).head(10))

,mean,median,std,q1,q3,iqr,spike_threshold_units,spike_observation_count
units_sold,9.432416,7.0,8.359259,4.0,13.0,9.0,40.0,2990


,product_id,category,mean_units,std_units,median_units,coefficient_of_variation,variability_rank
0,P001,Grocery,8.234795,8.340717,6.0,1.012863,1
30,P031,Electronics,7.617260,7.550303,6.0,0.991210,2
9,P010,Grocery,8.922329,8.820747,6.0,0.988615,3
14,P015,Home,8.548356,8.385492,6.0,0.980948,4
34,P035,Electronics,7.826301,7.546828,6.0,0.964291,5
2,P003,Grocery,8.021233,7.710206,6.0,0.961225,6
28,P029,Personal Care,9.480411,9.107720,7.0,0.960688,7
1,P002,Grocery,8.731370,8.380330,7.0,0.959796,8
7,P008,Grocery,9.598904,9.206239,7.0,0.959093,9
35,P036,Electronics,6.933699,6.636078,5.0,0.957076,10


## 4. Leakage-safe features

Demand lags and rolling statistics use shifted observations only. The calendar and price-change fields describe the forecast date; same-day promo and price are assumed known/planned at forecast time. Dates with insufficient lag history are omitted.

In [4]:
featured = add_features(panel)
print(f"Modeling rows after lag warm-up: {len(featured):,}")
display(featured[['date', 'store_id', 'product_id', 'units_sold', 'lag_7', 'rolling_mean_7', 'rolling_mean_30', 'holiday_flag', 'price_change_pct']].head())

Modeling rows after lag warm-up: 350,000


,date,store_id,product_id,units_sold,lag_7,rolling_mean_7,rolling_mean_30,holiday_flag,price_change_pct
0,2024-12-22,S01,P001,5,7.0,4.142857,2.866667,1,0.038757
1,2024-12-23,S01,P001,7,1.0,3.857143,3.033333,1,-0.053153
2,2024-12-24,S01,P001,8,5.0,4.714286,3.133333,1,0.024106
3,2024-12-25,S01,P001,3,4.0,5.142857,3.200000,1,-0.004407
4,2024-12-26,S01,P001,2,1.0,5.000000,3.200000,1,0.013603


## 5–6. Chronological training, tuning, and evaluation

The pipeline trains Linear Regression, Decision Tree, and Random Forest candidates. Tree parameters are selected using validation MAE only, then each selected model is refit on train+validation and evaluated on the held-out final interval. Metrics and product/store error slices are exported.

In [5]:
forecasts, metrics = fit_and_evaluate(featured)
display(metrics)
display(forecasts.head())
print(f"Detailed outputs: {PROCESSED_DIR}")

,model,validation_mae,train_end,validation_end,test_start,test_end,mae,rmse,r2
2,Random Forest,4.366253,2026-04-24,2026-08-08,2026-08-09,2026-11-21,3.739173,5.368611,0.398919
1,Decision Tree,4.423983,2026-04-24,2026-08-08,2026-08-09,2026-11-21,3.765839,5.425600,0.386090
0,Linear Regression,4.496468,2026-04-24,2026-08-08,2026-08-09,2026-11-21,3.886303,5.505598,0.367853


,forecast_date,store_id,product_id,actual_units,unit_price,promo_flag,forecast_units,model
0,2026-08-09,S01,P001,0,86.86,1,10.330652,Linear Regression
1,2026-08-10,S01,P001,4,92.30,0,1.207039,Linear Regression
2,2026-08-11,S01,P001,5,91.99,0,1.510023,Linear Regression
3,2026-08-12,S01,P001,2,85.68,1,6.426436,Linear Regression
4,2026-08-13,S01,P001,3,96.32,0,1.159679,Linear Regression


Detailed outputs: D:\cogniza internship data science project\project 2\data\processed


## 7. Inventory risk and data dictionary

Inventory risk uses the best test-MAE model's average forecast across its latest 28 test dates. Expected lead-time demand is average daily forecast multiplied by product lead time. Exposure is a planning estimate (not a measured lost-sales quantity).

In [6]:
products = pd.read_csv(RAW_DIR / 'products.csv')
risk = export_inventory_risk(forecasts, inventory, products)
export_data_dictionary()
display(risk.sort_values('reorder_priority_score', ascending=False).head(20))
print('Created: actuals_vs_forecasts.csv, forecast_metrics.csv, inventory_risk.csv, error slices, statistics, figures, and data dictionary.')

,store_id,product_id,current_stock_on_hand,reorder_point,lead_time_days,avg_daily_forecast_units,lead_time_demand_units,expected_stockout_units,stockout_exposure_flag,inventory_coverage_days,reorder_priority_score
438,S09,P039,9,107,21,4.964759,104.259934,95.259934,1,1.812777,91.367729
38,S01,P039,29,313,21,12.252861,257.310077,228.310077,1,2.366794,88.729551
402,S09,P003,6,47,16,3.279305,52.468879,46.468879,1,1.829656,88.564650
279,S06,P030,32,338,19,14.235906,270.482217,238.482217,1,2.247837,88.169278
324,S07,P025,18,182,19,7.990736,151.823979,133.823979,1,2.252609,88.144165
497,S10,P048,5,46,16,2.585971,41.375541,36.375541,1,1.933509,87.915566
329,S07,P030,19,167,19,7.363845,139.913064,120.913064,1,2.580174,86.420139
178,S04,P029,11,107,21,3.744605,78.636699,67.636699,1,2.937560,86.011620
424,S09,P025,32,267,19,11.947846,227.009079,195.009079,1,2.678307,85.903647
84,S02,P035,19,161,16,8.095256,129.524099,110.524099,1,2.347054,85.330915


Created: actuals_vs_forecasts.csv, forecast_metrics.csv, inventory_risk.csv, error slices, statistics, figures, and data dictionary.
